# 01 – Exploration
Free-form digging into the client sample.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 60)

df = pd.read_excel("../data/raw/presales_data_sample.xlsx")
df.shape

In [ ]:
# All candidates for one input, key columns side by side
cols = ["input_company_name", "input_main_country_code", "input_main_city",
        "company_name", "main_country_code", "main_city", "website_domain", "main_industry"]
df.loc[df.input_row_key == 2, cols]

## Step 1 – Manual review of a varied sample

20 inputs, stratified so each kind of case shows up:
- **nordics** (DK/NO/SE/FI, half the data): 8
- **asia** (SG/MY/PK/TH/CN/BD/IN): 6
- **other** countries: 3
- **no_candidate_in_country**: none of the 5 candidates is in the input's country (any region): 3

Fixed `random_state` so the sample is reproducible.

In [ ]:
NORDICS = {"DK", "NO", "SE", "FI"}
ASIA = {"SG", "MY", "PK", "TH", "CN", "BD", "IN"}

df["cand_in_input_country"] = df.main_country_code == df.input_main_country_code
inputs = df.groupby("input_row_key").agg(
    name=("input_company_name", "first"),
    country=("input_main_country_code", "first"),
    any_cand_in_country=("cand_in_input_country", "any"),
).reset_index()

inputs["group"] = "other"
inputs.loc[inputs.country.isin(NORDICS), "group"] = "nordics"
inputs.loc[inputs.country.isin(ASIA), "group"] = "asia"
inputs.loc[~inputs.any_cand_in_country, "group"] = "no_candidate_in_country"
inputs.group.value_counts()

In [ ]:
quota = {"nordics": 8, "asia": 6, "other": 3, "no_candidate_in_country": 3}
sample = pd.concat(
    inputs[inputs.group == g].sample(n, random_state=42) for g, n in quota.items()
).reset_index(drop=True)
keys = sample.input_row_key.tolist()
sample

In [7]:
INPUT_COLS = ["input_company_name", "input_main_country_code", "input_main_region",
              "input_main_city", "input_main_postcode", "input_main_street", "input_main_street_number"]
CAND_COLS = ["company_name", "company_legal_names", "main_country_code", "main_city",
             "main_postcode", "main_street", "website_domain", "main_industry", "employee_count"]

def show(i):
    """Show the i-th sampled input once, then its candidates."""
    rows = df[df.input_row_key == keys[i]]
    print(f"[{i}] key={keys[i]}  group={sample.group[i]}")
    display(rows[INPUT_COLS].head(1))
    display(rows[CAND_COLS].reset_index(drop=True))

show(2)   # change to 1, 2, ... 19

[2] key=168  group=nordics


,input_company_name,input_main_country_code,input_main_region,input_main_city,input_main_postcode,input_main_street,input_main_street_number
385,ENERGI HOBRO TELE A/S,DK,Central Denmark Region,Silkeborg,8600,Tietgensvej,4


,company_name,company_legal_names,main_country_code,main_city,main_postcode,main_street,website_domain,main_industry,employee_count
0,Energi Fyn,Energi Fyn Holding A/S. | Energi Fyn a.m.b.a,DK,Odense,5250,Sanderumvej,energifyn.dk,Renewable energy,150.0
1,Norlys,Norlys Energi A/S,DK,Silkeborg,8600,Tietgensvej,norlys.dk,Telecommunication Services,2500.0
2,Energimidt,EnergiMidt Handel A/S,DK,Silkeborg,8600,Tietgensvej,energimidt.dk,Utillities,NaN
3,Eniig Energi Naturgas,Eniig Fiber A/S. | Eniig Energi Naturgas A/S | Stofa Fib...,DK,Silkeborg,8600,Tietgensvej,eniig.com,Telecommunication Services,2500.0
4,Energi Danmark Securities A/S.,Energi Danmark Securities A/S.,DK,Aarhus,8200,Tangen,NaN,Utillities,NaN


### Notes (one line per input)

| # | key | my pick (company / none) | confidence | why |
|---|-----|--------------------------|------------|-----|
| 0 | 537 | cand 0: TINEX Group | high | exact name + street + number (Verkseier Furulunds Vei 1); cand 3 TINEX AS & cand 4 TINEX Globe AS = same group, different legal entities |
| 1 | 305 | cand 3: Leaseplan (SE, Solna) | med-high | only SE/Solna candidate, leaseplan.se; LeasePlan→Ayvens rebrand (2023), input name possibly outdated, check allabolag; QC: employees=1 implausible, industry empty |
| 2 | 168 | cand 3: Eniig (= Sinal A/S, CVR 36423544) | med | UNVERIFIED (name history from OpenCorporates, not checked on official CVR); input is a former name of Verdo Tele A/S, merged into Sinal A/S (Norlys group) 2022; Sinal was formerly Eniig Fiber A/S = cand 3 legal name (source: OpenCorporates); cands 1-3 share Norlys HQ address so address does not discriminate; QC: cand 3 conflates fibre co with Eniig Energi Naturgas A/S, outdated name + domain (eniig.com) |
| 3 |     |                          |            |     |

### Recurring patterns
- 